In [ ]:
import mne
from visualization.vis_eeg import ch_psd_subplots, ics_psd_subplots, compare_psds
from preprocessing.preprocess_eeg  import load_raw, prepro_for_ica, get_ica, get_ica_sources, plot_ics
from analysis.process_eeg import compute_psd

%load_ext autoreload
%autoreload 2

In [ ]:
test = False
show = False
save = True
load = False
auto_mode = False

# 1. Inspect raw data

## Load data

In [ ]:
cids_by_sid = {
    '01': ['RS_pre', 'RS_cTBS', 'SpaNav_task2'],
    '02': ['RS_EO', 'RS_EC', 'task_cTBS', 'task_HF', 'task_iTBS'],
}

In [ ]:
sid = '02'
cid = cids_by_sid[sid][4]
cid

In [ ]:
raw_start = load_raw(sid, cid, test)

## Mark noisy channels

In [ ]:
scroll_rec_kwargs = {
    # 'duration': 20, 
    'duration': raw_start.times[-1] if raw_start.times[-1] < 230 else raw_start.times[-1]//2,
    'n_channels': len(raw_start.ch_names),
    'clipping': None,
    'scalings': {'eeg': 6e-5},
}

In [ ]:
if not auto_mode:
    %matplotlib qt6
    raw_start.plot(**scroll_rec_kwargs, picks='eeg')

In [ ]:
raw_start.info['bads'] = ['O1']

## Compute and plot PSD

In [ ]:
psd_by_ch = compute_psd(raw_start, fmax=200, test=test)

In [ ]:
# Plot each channel PSD as subplot
%matplotlib inline
ch_psd_subplots(raw_start, psd_by_ch, cid=cid, sid=sid, test=test, save=save, show=show, file_name_suff='raw')

In [ ]:
%matplotlib inline
psd_by_ch.plot(picks=psd_by_ch.ch_names, average=False);

In [ ]:
if not auto_mode:
    # Interactive plot PSD by channels on topography-location
    %matplotlib qt6
    psd_by_ch.plot_topo(color='k', axis_facecolor='w', fig_facecolor='w', dB=True)  # does not show 'bads' !

# 2. Preprocess the data

In [ ]:
prepro_raw = prepro_for_ica(sid, cid, raw_start, load=True, save=False)

In [ ]:
%matplotlib qt6
scroll_preprorec_kwargs = {
    'duration': 20, 
    'n_channels': 60,
    'scalings': {'eeg': 1e-4}, 
}

In [ ]:
if not auto_mode:
    prepro_raw.plot(**scroll_preprorec_kwargs, picks='eeg')

# 3. Perform ICA

In [ ]:
n_components_based_on_var = True

In [ ]:
ica = get_ica(prepro_raw, n_components_based_on_var)

In [ ]:
ica_sources = get_ica_sources(ica, prepro_raw, sid, cid, load=load, save=save)

### Inspect ICA result

In [ ]:
%matplotlib inline
plot_ics(ica, prepro_raw, sid, cid, test=test, save=save, show=show)

In [ ]:
if not auto_mode:
    %matplotlib qt6
    ica.plot_sources(prepro_raw)

In [ ]:
ica_psd_fmin, ica_psd_fmax = prepro-raw.info['highpass'], prepro-raw.info['lowpass']
ica_psds = compute_psd(ica_sources, test=test, fmin=ica_psd_fmin, fmax=ica_psd_fmax)

In [ ]:
%matplotlib inline
ics_psd_subplots(ica_psds, test=test, save=save, show=show, cid=cid, sid=sid)

# Compare different recording sessions 

In [ ]:
cid1 = (cid, '/Volumes/My Passport/SpaNav/Sophie_backup/Data/Raw')
cid2 = (f'{cid}_ica', '/Volumes/My Passport/SpaNav/Sophie_backup/Data/RawClean')

In [ ]:
cids_dict = {}
psds_dict = {}

In [ ]:
for cid, cid_dir in [cid1, cid2]:
    rec = mne.io.read_raw_brainvision(vhdr_fname=f'{cid_dir}/practical_session_01/{cid}.vhdr')
    psds_dict[rec] = compute_psd(rec, test=test)

In [ ]:
compare_psds(psds_dict, test=True, show=False, save=True)